In [6]:
import numpy as np
import math
import random
import csv
import pandas as pd
import scipy
import scipy.optimize as opt

In [10]:
def read_behavioral_data(n):
    result_stay_cue = []
    result_safe_risk = []
    action_stay_cue = []
    action_safe_risk = []
    if_can_ask = []
    fname = './behavioral_data/uncertainty_' + str(n+1) + '_2022.csv'
    with open(fname,'r') as f :
        for line in f.readlines():
            if line.split(',')[0]=='0' or line.split(',')[0]=='1':
                if int(line.split(',')[3])==0:
                    action_stay_cue.append(0)
                elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
                    action_stay_cue.append(1)
                else :
                    print('ERROR')
                result_stay_cue.append(int(line.split(',')[3]))
                result_safe_risk.append(int(float(line.split(',')[6])))
                action_safe_risk.append(int(line.split(',')[4]))
                if line.split(',')[1]==' ':
                    if_can_ask.append(0)
                else :
                    if_can_ask.append(1)

    return if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk

In [19]:
def dir(a):
    a_0 = np.sum(a,axis=0)
    A = np.zeros([8,8])
    for i in range(np.shape(A)[0]):
        for j in range(np.shape(A)[1]):
            A[i,j] = a[i,j]/a_0[j]
    return A
def cum(a):
    a_0 = np.sum(a,axis=0)
    a_cum = np.array([np.ones([1,8])*a_0[0],
                      np.ones([1,8])*a_0[1],
                      np.ones([1,8])*a_0[2],
                      np.ones([1,8])*a_0[3],
                      np.ones([1,8])*a_0[4],
                      np.ones([1,8])*a_0[5],
                      np.ones([1,8])*a_0[6],
                      np.ones([1,8])*a_0[7]]).squeeze().T
    return a_cum
def H_entropy(A):
    H = np.matmul(A.T,np.log(A+np.e**(-16)))
    H = np.diag(H)
    return H
def G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex):
    a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
    s = np.array(s)
    discount = 0.1
    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    H = -np.dot(H_entropy(A),s.reshape(-1,1))
    AL = np.dot(o,np.dot(w,s.reshape(-1,1)))
    AI = H + np.dot(o,np.log(o+np.e**(-16)).reshape(-1,1))
    preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
    EX = np.dot(o,preference.reshape(-1,1))
    if action_stay_cue == 0:
        return float(discount*(p_al*AL + p_ai*AI) - p_ex*EX)
    elif action_stay_cue == 1:
        return float(p_al*AL + p_ai*AI - p_ex*EX)
    else :
        print('ERROR')
        
def P_stay_cue(A,a,action_stay_cue,if_can_ask,p_al,p_ai,p_ex):#stay,cue,0,stay-safe,1,stay-risk,2,cue-safe,3,cue-risk
    if if_can_ask == 0:
        return 1
    else:
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0.5,0.5,0,0,0,0,0,0])
        G_stay_safe = G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0,0,0.5,0.5,0,0,0,0])
        G_stay_risk = G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [1, 0, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 1, 0, 0, 0, 0, 0]
        G_cue_HR_0 = G_ExperctedFreeEnergy(A, a, s1, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s2, 1, p_al, p_ai, p_ex)
        G_cue_HR_1 = G_ExperctedFreeEnergy(A, a, s3, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s4, 1, p_al, p_ai, p_ex)
        if G_cue_HR_0>G_cue_HR_1:
            G_cue_HR = G_cue_HR_1
        else:
            G_cue_HR = G_cue_HR_0
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [0, 1, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 0, 1, 0, 0, 0, 0]
        G_cue_LR_0 = G_ExperctedFreeEnergy(A, a, s1, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s2, 1, p_al, p_ai, p_ex)
        G_cue_LR_1 = G_ExperctedFreeEnergy(A, a, s3, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s4, 1, p_al, p_ai, p_ex)
        if G_cue_LR_0>G_cue_LR_1:
            G_cue_LR = G_cue_LR_1
        else:
            G_cue_LR = G_cue_LR_0
        G_stay_cue = np.array([-min(G_stay_safe,G_stay_risk),-(G_cue_HR+G_cue_LR)/2])
        if G_stay_cue[0]>20:
            G_stay_cue[0]=20
        if G_stay_cue[1]>20:
            G_stay_cue[1]=20
        exp_G = [np.exp(G_stay_cue[0]),np.exp(G_stay_cue[1])]
        total = exp_G[0]+exp_G[1]
        P = [exp_G[0]/total,exp_G[1]/total]
        return P[action_stay_cue].squeeze()

def P_safe_risk(A,a,action_safe_risk,result_stay_cue,p_al,p_ai,p_ex):#con=0,no,con=1,HRC,con=2,LRC,pi=0,safe,pi=1,risky
    if result_stay_cue !=0:
        action_stay_cue = 1
    else :
        action_stay_cue = 0
    if result_stay_cue == 0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    elif action_safe_risk == 1:
        s=np.array([1,0,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,1,0,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    else :
        s=np.array([0,1,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,0,1,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    if G_safe > 20:
        G_safe = 20
    if G_risk > 20:
        G_risk = 20
    exp_G = [np.exp(G_safe),np.exp(G_risk)]
    total = exp_G[0]+exp_G[1]
    P_safe_risk = [exp_G[0]/total,exp_G[1]/total]
    return P_safe_risk[action_safe_risk].squeeze()

def a_update(a,result_stay_cue,result_safe_risk,action_safe_risk,rate):
    if result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
#safeHRC,safeLRC,riskyHRC,riskLRC,stayHRC,stayLRC,cueHRC,cueLRC
        o=np.array([1,0,0,0,0,0,0,0])
#safe,riskyHR,riskyLR,stay,cueHR,cueLR
    elif result_stay_cue==0 and result_safe_risk==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==3:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==9:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==12:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([1,0,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==0:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==3:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==9:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])      
    elif result_stay_cue==1 and result_safe_risk==12:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0,1,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==0:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==3:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==9:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])    
    else :
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    if result_stay_cue!=0:
        a=a+rate*np.outer(o,s)#rate:learning rate
    else :
        a=a+rate*0.1*np.outer(o,s)
    return a

def P_active_inference(x):
    trial_num = 120
    subject_num = 25
    if_can_ask_sub = []
    action_stay_cue_sub = []
    result_stay_cue_sub = []
    action_safe_risk_sub = []
    result_safe_risk_sub = []
    for i in range(subject_num):
        if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(i)
        if_can_ask_sub.append(if_can_ask)
        action_stay_cue_sub.append(action_stay_cue)
        result_stay_cue_sub.append(result_stay_cue)
        action_safe_risk_sub.append(action_safe_risk)
        result_safe_risk_sub.append(result_safe_risk)
    rate = []
    prior = []
    p_al = []
    p_ai = []
    p_ex = []
    for i in range(subject_num):
        rate.append(x[i*5])
        prior.append(x[i*5+1])
        p_al.append(x[i*5+2])
        p_ai.append(x[i*5+3])
        p_ex.append(x[i*5+4])
    neg_log_p_policy = 0
    for i in range(subject_num):
        a = np.array([[100.0,100,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
        A = dir(a)
        for j in range(trial_num):
            neg_log_p_policy -= np.log(P_stay_cue(A,a,action_stay_cue_sub[i][j],if_can_ask_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            neg_log_p_policy -= np.log(P_safe_risk(A,a,action_safe_risk_sub[i][j],result_stay_cue_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            a = a_update(a,result_stay_cue_sub[i][j],result_safe_risk_sub[i][j],action_safe_risk_sub[i][j],rate[i])
            A = dir(a)
    return neg_log_p_policy

In [20]:
subject_num = 25
trial_num = 120
x_0 = np.ones(subject_num*trial_num)
fitted_parameter =  opt.minimize(P_active_inference,x0=x_0,method='BFGS')

c:\Users\dell\miniconda3\envs\RL\lib\site-packages\ipykernel_launcher.py:20: RuntimeWarning: invalid value encountered in log
c:\Users\dell\miniconda3\envs\RL\lib\site-packages\ipykernel_launcher.py:39: RuntimeWarning: invalid value encountered in log
c:\Users\dell\miniconda3\envs\RL\lib\site-packages\ipykernel_launcher.py:219: RuntimeWarning: divide by zero encountered in log


KeyboardInterrupt: 